
# 🧪 Mini RAG Learning Lab — From PDF to Vector Search to Answer

**A single-notebook, learning-first walk through every step of Retrieval-Augmented Generation — built for Python/ML interview preparation.**

> ⏱️ Expected runtime: ~5–10 minutes on a free Google Colab CPU runtime (most of it is model downloads).
> 🚫 No LangChain. No hidden abstractions. You will see **every** intermediate value.

**The chain you will master in this notebook:**

```
PDF → text → chunks → tokens → token IDs → embedding lookup → vectors
→ Transformer/embedding model → final chunk embedding → vector database
→ query embedding → similarity search → retrieval → re-ranking → context → LLM answer
```



## 📖 README — What this notebook teaches & how to use it

### What this project teaches
Every stage of a RAG pipeline, **from the inside**: what a token ID actually is, where embedding
numbers come from (learned parameters + gradient descent), how self-attention computes contextual
vectors, what cosine similarity measures, what a vector DB does (and does NOT do), and how
retrieval → re-ranking → prompt → LLM fit together.

### Architecture
- **Ingestion (offline):** PDF → text extraction → chunking → embedding model → vector DB.
- **Query (online):** user question → query embedding → vector search → top-K → cross-encoder re-ranking → prompt → LLM → grounded answer with citations.

### Installation / setup
No local install needed. Upload this notebook to Google Colab (colab.research.google.com) and run
the cells **top to bottom**. The first code cell installs all dependencies:
`PyMuPDF`, `sentence-transformers`, `chromadb`, `scikit-learn`, `matplotlib`, `ipywidgets`, `pandas`.
Optional: an API key from any OpenAI-compatible provider (OpenCode Zen, OpenAI, Gemini, Groq,
or a custom base URL) for the final LLM cell (Section 18). Without a key the
notebook shows a clearly-labelled DEMO answer so the pipeline still completes.

### How to run
Run every cell in order. The notebook is linear by design — each section feeds the next.
The last cell is a full end-to-end smoke test of the whole pipeline.

### What each section demonstrates
| # | Section | # | Section |
|---|---------|---|---------|
| 1 | RAG overview | 14 | Vector database (ChromaDB) |
| 2 | PDF text extraction (PyMuPDF) | 15 | Top-K retrieval |
| 3 | Chunking + overlap | 16 | Cross-encoder re-ranking |
| 4 | Tokenization + token IDs | 17 | Building the RAG prompt |
| 5 | Token ID → embedding lookup (toy) | 18 | Final LLM answer (+ demo mode) |
| 6 | How embeddings are **learned** (PyTorch) | 19 | Visual RAG dashboard |
| 7 | Real embedding model (MiniLM, 384-d) | 20 | Interactive learning widgets |
| 8 | Inside the embedding model | 21 | RAG failure experiments |
| 9 | Self-attention mathematics (toy) | 22 | RAG vs fine-tuning |
| 10 | Pooling → chunk embedding | 23 | Full architecture recap |
| 11 | Vector space + PCA plot | 24 | Code quality + LangChain mapping |
| 12 | Query embedding | 25 | Interview checkpoints + cheat sheet + glossary |
| 13 | Cosine similarity (by hand + real) | | |

### Toy mathematics vs real model behavior
Cells that use small hand-made numbers are always labelled **`🧸 TOY EXAMPLE — NOT REAL MODEL OUTPUT`**.
Cells that print values produced by a real model are labelled **`🎯 ACTUAL MODEL OUTPUT`**.
Never mix the two when you explain this in an interview.

### How to explain this project in an interview
"Documents are chunked and embedded once at ingestion time into a vector database. At query time
the question is embedded with the same model, the DB returns the top-K chunks by cosine similarity,
a cross-encoder re-ranks those candidates for precise relevance, and the best chunks are placed
into a prompt so the LLM answers from grounded context instead of memory."


In [ ]:

# ⚙️ SETUP — run this cell once (fresh Colab runtime: takes ~1-2 min, mostly downloads)
#   PyMuPDF (fitz)        -> create + read PDFs
#   sentence-transformers -> real embedding model + real cross-encoder re-ranker
#   chromadb              -> local vector database
#   scikit-learn          -> PCA + cosine similarity helpers
#   matplotlib / pandas / ipywidgets -> visuals and interactivity

import subprocess, sys

def install_if_missing(pkg, import_name=None):
    """Install a package only if it is not already importable."""
    try:
        __import__(import_name or pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, imp in [("pymupdf", "fitz"), ("sentence-transformers", "sentence_transformers"),
               ("chromadb", "chromadb"), ("scikit-learn", "sklearn"),
               ("matplotlib", "matplotlib"), ("ipywidgets", "ipywidgets"),
               ("pandas", "pandas")]:
    install_if_missing(p, imp)

print("All dependencies ready.\n")

import getpass
import json
import os
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymupdf  # PyMuPDF

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 60)

print("Environment ready")
print("   numpy", np.__version__, "| PyMuPDF", pymupdf.__doc__[:50] if pymupdf.__doc__ else "loaded")



---

# 📌 SECTION 1 — RAG OVERVIEW

## 📌 Simple definition
**RAG = Retrieval + Augmented + Generation.**

Instead of asking an LLM to answer from its (possibly stale or wrong) memory, we first **retrieve**
relevant documents, **augment** the prompt with that retrieved context, and let the LLM
**generate** an answer grounded in that context.

## 🧠 Intuition
An LLM is like an expert giving a closed-book exam. RAG hands the expert an **open-book exam**
with the exact pages they need.

## 🔢 The high-level flow
```
USER QUESTION
      ↓
RETRIEVAL   (find the most relevant chunks of our documents)
      ↓
RELEVANT CONTEXT   (the retrieved chunks, pasted into the prompt)
      ↓
LLM         (reads question + context, writes an answer)
      ↓
ANSWER      (grounded in the supplied context, not in fuzzy memory)
```

## 💡 What we just learned
RAG has **two phases**:
- **Ingestion (offline):** documents → chunks → embeddings → vector DB.
- **Query (online):** question → embedding → search → retrieve → re-rank → prompt → LLM.

Keep that separation in your head — we will build ingestion first, then query time.

## 🎤 Interview checkpoint
**Q: What is RAG and why do we need it?**
**A:** RAG retrieves relevant external context and injects it into the LLM prompt, so the answer
is grounded in up-to-date, domain-specific information. It reduces hallucination and avoids
expensive fine-tuning when knowledge changes frequently.



---

# 📌 SECTION 2 — PDF TEXT EXTRACTION

## 📌 Simple definition
Before a machine can do anything with a document, the text must be pulled out of the binary PDF
format and turned into a plain Python string.

## 🔢 The transformation
```
PDF (binary file)
      ↓  PDF parser (PyMuPDF = fitz)
pages   → list of Page objects (layout, coordinates, fonts...)
      ↓  page.get_text()
text    → plain string per page
```

## 💻 Step 1 — Create our tiny document: `leave_policy.pdf`
We generate the PDF **programmatically** so the notebook is fully self-contained.


In [ ]:

# The whole "document corpus" of this project: ONE small leave-policy PDF.
POLICY_TEXT = """COMPANY LEAVE POLICY

Annual Leave:
Employees receive 18 days of annual leave per year.

Sick Leave:
Employees receive 10 days of paid sick leave per year.

Maternity Leave:
Female employees receive 26 weeks of paid maternity leave.

Paternity Leave:
Employees receive 7 days of paid paternity leave.

Work From Home:
Employees may work remotely up to 2 days per week with manager approval.
"""

def create_leave_policy_pdf(path="leave_policy.pdf", text=POLICY_TEXT):
    # Create a real PDF file with PyMuPDF
    doc = pymupdf.open()
    page = doc.new_page()                                    # one A4 page
    rect = pymupdf.Rect(72, 72, page.rect.width - 72, page.rect.height - 72)
    page.insert_textbox(rect, text, fontsize=13, fontname="helv")
    doc.save(path)
    doc.close()
    return path

pdf_path = create_leave_policy_pdf()
print(f"📄 Created {pdf_path!r} ({os.path.getsize(pdf_path):,} bytes, real PDF on disk)")



## 💻 Step 2 — Extract text + metadata

**Metadata matters for citations.** At the end of the pipeline we want to say
*"Answer based on `leave_policy.pdf`, page 1"* — so every piece of text must carry its source
with it through the entire journey.


In [ ]:

def extract_pdf(path):
    # PDF -> list of document objects: {'text': str, 'metadata': dict}
    doc = pymupdf.open(path)
    pages = []
    for page_no, page in enumerate(doc, start=1):
        pages.append({
            "text": page.get_text().strip(),
            "metadata": {
                "source": os.path.basename(path),
                "page": page_no,
            },
        })
    doc.close()
    return pages

pages = extract_pdf(pdf_path)
print(f"🎯 ACTUAL OUTPUT — extracted {len(pages)} page(s)\n")
for p in pages:
    print(json.dumps(p, indent=2))



## 📊 What we see
- The PDF contains **1 page**, so our corpus is a single document object.
- `text` holds the raw characters; `metadata` records `source` + `page`.
- Real documents would give one object per page (often with more metadata: section, title, date).

## 💡 What we just learned
- A PDF is **not** text — it is drawing instructions. A parser reconstructs the characters.
- Extraction output = **text + metadata**. Metadata is what makes final-answer citations possible.

## 🎤 Interview checkpoint
**Q: Why do we keep metadata alongside text in a RAG pipeline?**
**A:** So the system can cite where each piece of retrieved context came from (source file, page)
and so we can filter/index retrieval by document attributes.



---

# 📌 SECTION 3 — CHUNKING

## 📌 Simple definition
A **chunk** is a short, self-contained piece of text that we will later embed and retrieve.
We split the extracted text into chunks because (a) embedding models have a limited input window,
and (b) retrieval works better when each stored piece covers **one idea**.

## 🧠 Intuition
A search engine does not return a whole 400-page book for your question — it returns the few
paragraphs that matter. Chunks are the "paragraphs" of our vector database.

## 🔢 The transformation
```
FULL DOCUMENT
     ↓  chunking
Chunk 1
Chunk 2
Chunk 3
...
```

## 🔢 Why chunk size is a trade-off (no universal optimum!)
- **Too small** → a chunk may not contain enough context to answer a question ("26 weeks" split from "maternity leave").
- **Too large** → a chunk mixes many topics, so similarity scores become diluted and retrieval is less precise.

## 🔢 Chunk overlap
When we cut text at fixed boundaries, we can slice through an important phrase.
Overlap keeps a few words of the previous chunk at the start of the next one:

```
words:        A B C D E F G H
chunk 1:      A B C D E F G H
chunk 2:              E F G H I J K L
                     └─ overlap ─┘
```

Overlap preserves boundary context: a phrase split between chunks still appears **complete** in at least one chunk.

## 💻 Our simple chunker (no magic — plain Python)


In [ ]:

def chunk_text(text, chunk_size_words=20, overlap_words=4):
    """
    Naive word-based chunker.
    - Moves a sliding window of `chunk_size_words` words over the text.
    - Each next window starts `overlap_words` words BEFORE the previous window ended.
    """
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i : i + chunk_size_words]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if i + chunk_size_words >= len(words):
            break                      # last window reached the end
        i += chunk_size_words - overlap_words
    return chunks

# Merge all extracted pages into one text, then chunk.
full_text = " ".join(p["text"] for p in pages)

CHUNK_SIZE, CHUNK_OVERLAP = 20, 4
chunks = chunk_text(full_text, chunk_size_words=CHUNK_SIZE, overlap_words=CHUNK_OVERLAP)

print(f"🎯 ACTUAL OUTPUT — {len(chunks)} chunks (chunk_size={CHUNK_SIZE} words, overlap={CHUNK_OVERLAP} words)\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ({len(c.split())} words) ---")
    print(c)
    print()


In [ ]:

# 📊 Visualize which words belong to which chunk (overlap = darker overlap regions)
words = full_text.split()
fig, ax = plt.subplots(figsize=(13, 3))
colors = plt.cm.Set2(np.linspace(0, 1, len(chunks)))
for ci, c in enumerate(chunks):
    cwords = c.split()
    start = None
    # locate each chunk's word span in the full text (first occurrence of its first 3 words)
    for j in range(len(words) - len(cwords) + 1):
        if words[j:j+3] == cwords[:3]:
            start = j
            break
    ax.barh(ci, len(cwords), left=start, height=0.6, color=colors[ci], alpha=0.55,
            label=f"chunk {ci}")
    ax.text(start + len(cwords)/2, ci, f"chunk {ci}", ha="center", va="center", fontsize=9)
ax.set_yticks(range(len(chunks)))
ax.set_xlabel("word position in document")
ax.set_title("Chunk coverage of the document — overlapping regions appear stacked")
ax.set_xlim(0, len(words))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()



## 💡 What we just learned
- Chunking converts **one long text** into **retrievable units**.
- Overlap guarantees that phrases near boundaries are not cut into unusable pieces.
- There is no magic chunk size; it depends on the document and the model window.

## 🎤 Interview checkpoint
**Q: What happens if chunks are too small or too large?**
**A:** Too small → missing context, retrieval can return fragments that cannot answer the question.
Too large → diluted topic focus, lower similarity precision, and wasted prompt tokens.



---

# 📌 SECTION 4 — TOKENIZATION

## 📌 Simple definition
A **tokenizer** converts raw text into **tokens** — the atomic pieces the model's vocabulary knows —
and maps each token to a **token ID**, an integer index into the vocabulary table.

## 🧠 Intuition (the part people get confused about)
```
TEXT        "Female employees receive 26 weeks of paid maternity leave."
   ↓  TOKENIZER (rule-based: lowercase, split on punctuation, merge subwords)
TOKENS      ['female', 'employees', 'receive', '26', 'weeks', 'of', 'paid', 'maternity', 'leave', '.']
   ↓  VOCABULARY LOOKUP (dictionary: token string -> integer position)
TOKEN IDS   [2931, 5126, 4374, ...]   ← INTEGER POSITIONS, NOT MEANINGS!  (real MiniLM IDs)
```

**`"maternity" → 23676` does NOT mean "23676 = maternity".**
It means: the tokenizer's vocabulary is an ordered list of 30,522 tokens, the string
`"maternity"` happens to sit at position 23676 in that list (that is the REAL position in the
MiniLM vocabulary — verify it in the cell output below). Exactly like `"apple"` → page 47 of a dictionary:
47 is not the *meaning* of apple, it's just **where it is written**.

- **token ≠ word**: the model may split one word into several subword tokens (`"unbelievably"` → `['un', '##bel', '##ie', '##va', '##bly']` — real MiniLM output).
- **token ID ≠ meaning**: it is a vocabulary index used to look up a **learned vector** (next sections).
- Every model has its **own** tokenizer and its **own** vocabulary. IDs are meaningless across models.

## 💻 Use the ACTUAL tokenizer of our embedding model
The model we will use for embeddings is `sentence-transformers/all-MiniLM-L6-v2`.
Let's load its real tokenizer and inspect it.


In [ ]:

from transformers import AutoTokenizer

TOKENIZER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)   # downloads the real vocab file

print("🎯 ACTUAL MODEL OUTPUT — tokenizer facts")
print("  tokenizer    :", tokenizer.name_or_path)
print("  class        :", type(tokenizer).__name__, "(BERT-style WordPiece tokenizer)")
print("  vocabulary size:", len(tokenizer), "tokens")
print("  special tokens: CLS id =", tokenizer.cls_token_id, "| SEP id =", tokenizer.sep_token_id,
      "| PAD id =", tokenizer.pad_token_id, "| UNK id =", tokenizer.unk_token_id)


In [ ]:

# 🎯 Tokenize the REAL chunk from our document — no invented values, this prints live output.
text = "Female employees receive 26 weeks of paid maternity leave."

tokens = tokenizer.tokenize(text)                       # string -> token strings
ids    = tokenizer.convert_tokens_to_ids(tokens)        # token strings -> integer IDs

print("🎯 ACTUAL OUTPUT — token | token ID")
print(f"{'token':<15} {'id':>6}")
print("-" * 24)
for t, i in zip(tokens, ids):
    print(f"{t!r:<15} {i:>6}")
print("-" * 24)
print(f"{'TOTAL':<15} {len(tokens):>6} tokens")
print()
print("round-trip check  :", tokenizer.convert_tokens_to_string(tokens))
print("decode ids -> text:", tokenizer.decode(ids))


In [ ]:

# 🎯 How the model ACTUALLY receives the sentence (with special tokens)
encoded = tokenizer(text)                     # adds [CLS] at start and [SEP] at end
print("🎯 ACTUAL OUTPUT — tokenizer.__call__ output")
print(json.dumps(encoded, indent=2, default=str))
print()
print("tokens with specials:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("input_ids          :", encoded["input_ids"])
print("attention_mask     :", encoded["attention_mask"], "  <- 1 = real token, 0 = padding (none needed here)")


In [ ]:

# 🎯 Subword tokenization — one English word can become MULTIPLE tokens
for w in ["maternity", "unbelievably", "paternity", "remotely", "1000"]:
    toks = tokenizer.tokenize(w)
    print(f"{w!r:20} -> {toks}")


In [ ]:

# 🎯 Inspect the vocabulary itself: a plain Python dict {token_string: id}
vocab = tokenizer.get_vocab()
print("🎯 ACTUAL OUTPUT — vocabulary peek")
print("  type:", type(vocab), "| size:", len(vocab))
print("  vocab['maternity'] =", vocab.get("maternity", "not a whole token — try subwords"))
print("  vocab['##ably']    =", vocab.get("##ably", "not in vocab"))
print("  vocab['[CLS]']     =", vocab["[CLS]"], " (special token)")
print()
print("  first 5 entries :", list(vocab.items())[:5])
print("  entry #10000    :", list(vocab.items())[10000])


In [ ]:

# 🧪 Interactive: type any sentence, see its REAL tokens and IDs
from ipywidgets import interact

@interact(text="Female employees receive 26 weeks of paid maternity leave.")
def show_tokens(text):
    ts  = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(ts)
    if not ts:
        print("(no tokens)")
        return
    for t, i in zip(ts, ids):
        print(f"{t!r:<20} -> {i}")
    print(f"\n{len(ts)} tokens | decode check:", tokenizer.convert_tokens_to_string(ts))



## 📊 What changed
`"Female employees ... leave."` (a 44-character string) became a list of 10 token strings, then a
list of 10 integers. Nothing "semantic" happened — this is a pure **dictionary lookup**, like
translating a word into its page number in a paper dictionary.

## 💡 What we just learned
- The token ID is an **index**, not a value. `8084` carries zero meaning by itself.
- Tokenizers are deterministic: same text → same IDs, on any machine.
- Subword tokens (`##ably`) let the model represent rare/unknown words from pieces it knows.

## 🎤 Interview checkpoint
**Q: What is a token ID?**
**A:** An integer index that identifies a token inside the tokenizer's fixed vocabulary. It is used
to look up the token's learned embedding vector. It has no semantic meaning by itself.



---

# 📌 SECTION 5 — TOKEN ID → EMBEDDING LOOKUP (the missing link)

## 📌 Simple definition
An **embedding matrix** is a big table: **rows = vocabulary tokens, columns = embedding
dimensions**. Converting a token ID into a vector is simply **selecting one row** of that table.

## 🧸 TOY EXAMPLE — NOT REAL MODEL OUTPUT
Vocabulary size = 5, embedding dimension = 3:

```
              d1    d2    d3
E =  row 0  [0.0,  0.0,  0.0 ]   <- "<PAD>"
     row 1  [1.1,  0.3, -0.2 ]   <- "cat"
     row 2  [1.0, -0.1,  0.4 ]   <- "dog"
     row 3  [0.4,  0.8, -0.5 ]   <- "female"
     row 4  [0.2,  0.6,  0.9 ]   <- "maternity"
```

- **rows = vocabulary tokens** (5 of them)
- **columns = embedding dimensions** (3 learned numbers per token)

## 🔢 The lookup — two mathematically equivalent ways
**Way 1 — direct row lookup:**
```
token_id = 4  →  vector = E[4, :] = [0.2, 0.6, 0.9]
```

**Way 2 — one-hot × matrix:**
```
one_hot(4) = [0, 0, 0, 0, 1]        (1 at position 4, 0 everywhere else)

[0, 0, 0, 0, 1] × E  =  0·row0 + 0·row1 + 0·row2 + 0·row3 + 1·row4
                      =  row4
                      =  [0.2, 0.6, 0.9]
```

Modern frameworks use **Way 1** (efficient row lookup — `nn.Embedding` is literally an indexed
table). Way 2 is the mathematical definition that makes backpropagation theory clean: the one-hot
"selects" a row through matrix multiplication.


In [ ]:

import numpy as np

# 🧸 TOY embedding matrix: 5 vocabulary rows x 3 dimensions (hand-made numbers!)
TOY_VOCAB = ["<PAD>", "cat", "dog", "female", "maternity"]
TOY_E = np.array([
    [0.0,  0.0,  0.0],   # row 0 = <PAD>
    [1.1,  0.3, -0.2],   # row 1 = cat
    [1.0, -0.1,  0.4],   # row 2 = dog
    [0.4,  0.8, -0.5],   # row 3 = female
    [0.2,  0.6,  0.9],   # row 4 = maternity
])
print("🧸 TOY EXAMPLE ONLY — E shape:", TOY_E.shape, "(vocab=5 rows, dim=3 cols)")

token_id = 4   # "maternity" in this toy vocab
print(f"\n1) Direct row lookup:  E[{token_id}] = {TOY_E[token_id]}")

one_hot = np.zeros(len(TOY_VOCAB))
one_hot[token_id] = 1.0
print(f"2) one_hot({token_id}) = {one_hot}")

selected = one_hot @ TOY_E          # (1,5) x (5,3) -> (1,3)
print(f"3) one_hot @ E        = {selected}")

assert np.allclose(selected, TOY_E[token_id]), "the two ways MUST agree"
print("\n✅ Both ways produce the SAME vector (they are mathematically identical).")


In [ ]:

# 📊 Visualize the lookup: highlight the selected row of the embedding matrix
fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(TOY_E, cmap="RdYlGn", vmin=-0.6, vmax=1.2)
ax.set_xticks(range(3)); ax.set_xticklabels(["d1", "d2", "d3"])
ax.set_yticks(range(5)); ax.set_yticklabels(TOY_VOCAB)
for r in range(5):
    for c in range(3):
        ax.text(c, r, f"{TOY_E[r, c]:.1f}", ha="center", va="center",
                fontweight="bold" if r == token_id else "normal",
                color="black" if r != token_id else "white")
ax.add_patch(plt.Rectangle((-0.5, token_id - 0.5), 3, 1, fill=False,
                           edgecolor="blue", lw=3, label=f"row selected by id={token_id}"))
ax.set_title("🧸 TOY embedding matrix — one token ID = one row")
ax.legend(loc="lower right")
plt.colorbar(im, ax=ax, shrink=0.8, label="value")
plt.tight_layout()
plt.show()



## 💡 What we just learned
- "Embedding a token" = **table lookup**, nothing more.
- One-hot × E is the math-formal version of "pick row `token_id`".
- The values in the table are not assigned by a human — they are **learned** (next section!).

## 🎤 Interview checkpoint
**Q: What is an embedding matrix?**
**A:** A parameter matrix of shape `(vocab_size, embedding_dim)`. Each row is the learned vector
of one vocabulary token. Input token IDs index directly into its rows.



---

# 📌 SECTION 6 — WHERE DO EMBEDDING VALUES COME FROM? (they are LEARNED)

## 📌 Simple definition
Nobody ever decided that "d1 = 0.17 for this token". The embedding matrix is just another set of
**model parameters**, initialized randomly and **updated by gradient descent** during training so
the numbers become useful for the task (e.g., predicting nearby words).

## 🔢 The learning loop
```
initial embedding matrix  E
     ↓ forward pass       (lookup rows, compute a prediction)
prediction
     ↓ loss               (how wrong is the prediction?)
loss (a scalar number)
     ↓ backpropagation    (chain rule: how much did each number in E contribute to the error?)
gradient dL/dE
     ↓ gradient descent   E_new = E_old − learning_rate × dL/dE
updated embedding matrix
```

**Every symbol:** `L` = loss, `E` = embedding matrix, `dL/dE` = gradient of loss w.r.t. each entry,
`learning_rate` = step size (how far we move per update). The minus sign means we move **against**
the gradient — downhill in error space.

## 💻 Real PyTorch calculation (toy-sized, but REAL autograd numbers)
Task (toy): a tiny model reads 3 word IDs, averages their embedding rows, and predicts a
target vector. We print the real gradient and the real updated matrix.


In [ ]:

import torch

# 🧸 TOY EXAMPLE — 5-token vocab, 3-dim embeddings. Values are random init + REAL gradient math.
torch.manual_seed(0)
embed = torch.nn.Embedding(num_embeddings=5, embedding_dim=3)   # the learned table E
E_before = embed.weight.detach().clone()
print("🧸 TOY — initial E (random parameters):\n", E_before)

# "Training example": input ids [4, 1, 2]  (toy: maternity, cat, dog)
# target: a toy label vector we pretend is correct
ids    = torch.tensor([4, 1, 2])
target = torch.tensor([0.5, -0.2, 0.9])   # 🧸 toy label, NOT a real task

# ---- forward pass ----
x    = embed(ids)              # (3, 3): row lookup for each id
pred = x.mean(dim=0)           # (3,): mean-pool the 3 token vectors
loss = ((pred - target) ** 2).mean()     # MSE loss: mean of squared errors

print(f"\nforward:  ids={ids.tolist()} -> pooled prediction {pred.tolist()}")
print(f"loss    : {loss.item():.4f}  (MSE)")

# ---- backward pass: compute dL/dE ----
loss.backward()
grad = embed.weight.grad      # same shape as E: (5, 3)
print("\ngradient dL/dE (real autograd output):\n", grad)
print("\nNOTE: only rows that were USED in the forward pass (4,1,2) have non-zero gradients!")

# ---- gradient descent: E_new = E_old - lr * grad ----
lr = 0.1
with torch.no_grad():
    embed.weight -= lr * grad          # THE parameter update, by hand
E_after = embed.weight.detach().clone()

expected = E_before - lr * grad
print("\nupdated E (real):\n", E_after)
print("\n✅ E_after == E_before - lr * grad ?", torch.allclose(E_after, expected))



## 💡 What we just learned
- Embedding values are **parameters** — random at first, improved by gradient descent.
- The update rule is exactly the classic one: `E_new = E_old − lr · dL/dE`.
- Only rows actually used in a training step get gradient updates (sparse updates).
- **Distributed representations:** no single dimension "means" a concept. Meaning lives in the
  **pattern across all dimensions**, learned jointly with everything else in the model.

## 🎤 Interview checkpoint
**Q: Who chooses the numbers inside an embedding vector?**
**A:** No one directly. They are learned model parameters optimized by gradient descent so the
vectors support the training objective. This is why embeddings encode semantic relationships
(similar words end up with similar vectors) even though no dimension is labeled by a human.



---

# 📌 SECTION 8 — WHAT HAPPENS INSIDE A REAL EMBEDDING MODEL

(We present this conceptually now; Section 9 does the attention math, Section 10 verifies the
pooling against the real model's actual output.)

## 🔢 The full conceptual flow
```
TEXT  "Female employees receive 26 weeks of paid maternity leave."
   ↓  TOKENIZATION (Section 4)
TOKEN IDs  [2931, 5126, 4374, ...]
   ↓  EMBEDDING LOOKUP (Section 5)  -> one static row per token
TOKEN EMBEDDINGS  (10, 384)   <- each row: the token's dictionary entry, no context yet
   ↓  TRANSFORMER (12 stacked layers of self-attention + feed-forward)
   ↓  each token vector is UPDATED by looking at all the other tokens in the sentence
CONTEXTUAL REPRESENTATIONS  (10, 384)  <- "maternity" now influenced by "female", "weeks", ...
   ↓  POOLING (mean of all token vectors, weighted by the attention mask)
   ↓  optional PROJECTION / normalization
FINAL EMBEDDING  (384,)  <- one vector for the WHOLE sentence
```

## 🧠 Intuition
- The embedding lookup gives each word its **dictionary definition** (same vector everywhere).
- The Transformer layers make each word's vector **context-aware**: the vector of *"bank"* in
  *"river bank"* moves toward water; in *"bank account"* it moves toward finance.
- Pooling collapses 10 contextual vectors into **one** fixed-size vector describing the whole sentence.



---

# 📌 SECTION 9 — SELF-ATTENTION MATHEMATICS (toy, step by step)

## 📌 Simple definition
Self-attention lets each token gather information from every other token, weighted by how
**relevant** they are to each other. The formula:

```
Attention(Q, K, V) = softmax( Q·Kᵀ / √d_k ) · V

Q = X·W_Q      queries:  "what am I looking for?"
K = X·W_K      keys:     "what do I contain?"
V = X·W_V      values:   "what information do I offer?"
```

1. `Q·Kᵀ` → compatibility score between every pair of tokens.
2. `/√d_k` → scaling to keep softmax gradients healthy.
3. `softmax` → turns each row of scores into weights that sum to 1.
4. `·V` → each output row = weighted average of all value vectors.

## 🧸 TOY ATTENTION CALCULATION FOR LEARNING — NOT REAL MODEL OUTPUT
3 tokens, `d_model = 4`, `d_k = 2`. Small random weights, then every intermediate printed.


In [ ]:

# 🧸 TOY ATTENTION — hand-built with NumPy, every step printed. NOT real model output.
rng = np.random.default_rng(42)
d_model, d_k, d_v = 4, 2, 2

X = np.array([                       # 3 toy "token vectors" (3 tokens, 4 dims each)
    [1.0, 0.0, 1.0, 0.0],            # token 0
    [0.0, 1.0, 0.0, 1.0],            # token 1
    [1.0, 1.0, 0.0, 0.0],            # token 2
])
W_Q = rng.normal(0, 1, (d_model, d_k))
W_K = rng.normal(0, 1, (d_model, d_k))
W_V = rng.normal(0, 1, (d_model, d_v))

Q = X @ W_Q     # (3,2)
K = X @ W_K     # (3,2)
V = X @ W_V     # (3,2)
print("🧸 Q = X·W_Q:\n", np.round(Q, 3))
print("🧸 K = X·W_K:\n", np.round(K, 3))
print("🧸 V = X·W_V:\n", np.round(V, 3))

scores = (Q @ K.T) / np.sqrt(d_k)    # (3,3) compatibility matrix, scaled
print("\n🧸 Q·Kᵀ / √d_k  (scaled scores):\n", np.round(scores, 3))

def softmax_rows(a):
    e = np.exp(a - a.max(axis=1, keepdims=True))   # subtract max for numerical stability
    return e / e.sum(axis=1, keepdims=True)

A = softmax_rows(scores)             # (3,3) attention weights, rows sum to 1
print("\n🧸 attention weights A = softmax(scores):\n", np.round(A, 3))
print("   row sums:", A.sum(axis=1), " (each row sums to 1 -> a weighted average)")

out = A @ V                          # (3,3)x(3,2) -> (3,2)
print("\n🧸 output = A·V  (each row = weighted sum of ALL value vectors):\n", np.round(out, 3))


In [ ]:

# 📊 Visualize the toy attention matrix
fig, ax = plt.subplots(figsize=(4.5, 3.5))
im = ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(["token0", "token1", "token2"])
ax.set_yticklabels(["token0", "token1", "token2"])
ax.set_xlabel("attended TO (key)")
ax.set_ylabel("attends FROM (query)")
for r in range(3):
    for c in range(3):
        ax.text(c, r, f"{A[r, c]:.2f}", ha="center", va="center")
ax.set_title("🧸 TOY attention weights — one row per query token")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()



## 💡 What we just learned
- Attention output for token *i* = **weighted average of all tokens' values**, weights = how
  compatible token *i* is with each other token.
- That is what makes representations **contextual**: the same word gets a different vector
  depending on its sentence, because attention mixes in its neighbours.
- A real MiniLM has **12 layers** of this, each with 12 attention heads, on 384-dim vectors.

## 🎤 Interview checkpoint
**Q: What are Q, K, V in self-attention?**
**A:** Learned linear projections of the input: queries express "what I'm looking for", keys
"what I contain", values "what I communicate". Compatibility (Q·Kᵀ) is normalized with softmax
into weights that average the values.



---

# 📌 SECTION 7 — THE REAL EMBEDDING MODEL

## 📌 Simple definition
Now we leave toy land. We load a **real, pretrained sentence-embedding model** and feed it real
text. The model downloads (~90 MB) and runs on Colab's CPU.

- Model: **`sentence-transformers/all-MiniLM-L6-v2`**
- Family: MiniLM, a distilled BERT-style Transformer, 12 layers
- Vocabulary: 30,522 tokens (the tokenizer from Section 4)
- Embedding dimension: **384**
- Pooling: **mean of token vectors** (we verify this in Section 10)

```
chunk  "Female employees receive 26 weeks of paid maternity leave."
   ↓  real embedding model (Transformer + mean pooling + normalization)
actual vector  ->  shape (384,)
```


In [ ]:

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
try:
    model = SentenceTransformer(MODEL_NAME)      # first call downloads the weights
except Exception as e:
    print(f"WARNING: Could not load {MODEL_NAME}: {e}")
    print("Trying fallback model: sentence-transformers/paraphrase-MiniLM-L6-v2")
    MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L6-v2"
    model = SentenceTransformer(MODEL_NAME)

print("🎯 ACTUAL MODEL OUTPUT — model facts")
# sentence-transformers >=5 renamed this getter; support both names
_dim_fn = getattr(model, "get_embedding_dimension", None) or model.get_sentence_embedding_dimension
print("  model                     :", MODEL_NAME)
print("  sentence embedding dim    :", _dim_fn())
print(model)   # sentence-transformers prints its own architecture summary (layers, pooling)


In [ ]:

# 🎯 REAL embedding of the REAL chunk. Values below are produced by the model AT RUNTIME.
text = "Female employees receive 26 weeks of paid maternity leave."
emb = model.encode(text, normalize_embeddings=True)     # (384,) L2-normalized

print("🎯 ACTUAL MODEL OUTPUT")
print("  input text       :", text)
print("  embedding shape  :", emb.shape)
print("  first 10 values  :", np.round(emb[:10], 6))
print("  L2 norm          :", round(float(np.linalg.norm(emb)), 6),
      "  (1.0 because we normalized — direction only, magnitude removed)")
print("  min / max        :", round(float(emb.min()), 4), "/", round(float(emb.max()), 4))
print()
print("⚠️  These 384 numbers ARE the sentence, as far as the vector space is concerned.")
print("⚠️  Dimension #7 does NOT mean 'maternity'. Meaning is distributed across all 384.")



## 💡 What we just learned
- The real model maps any sentence to a **384-dimensional vector**.
- We normalize to unit length so only **direction** matters (this makes cosine similarity natural).
- The vector is opaque to humans — but sentences with similar meaning end up with similar vectors.

## 🎤 Interview checkpoint
**Q: What is the embedding dimension and why does it matter?**
**A:** It is the size of the vector representing each text (384 for MiniLM). Higher dimensions can
store more information but cost more memory/compute; the dimension is fixed by the model architecture.



---

# 📌 SECTION 10 — POOLING: TOKEN VECTORS → ONE CHUNK VECTOR

## 📌 Simple definition
The Transformer outputs one 384-d vector **per token**. We need **one** vector for the whole
chunk. **Pooling** summarizes the token vectors into a single fixed-size vector.
MiniLM uses **mean pooling**: average all token vectors (ignoring padding), then L2-normalize.

## 🔢 The transformation
```
token representations  (T, 384)
     ↓  mean over the T tokens (only real tokens, mask out padding)
one vector             (384,)
     ↓  L2 normalization  v / ||v||
final chunk embedding  (384,)   <- exactly what model.encode() returns
```

## 💻 Let's PROVE it — run the model's insides by hand and compare
We use the same tokenizer, the same Transformer weights, do mean pooling ourselves with NumPy,
and check whether our hand-made result **equals** `model.encode()`. If it matches, we have truly
seen what happens inside. (We get the weights via `model[0].auto_model` — the underlying HF model.)


In [ ]:

# 🎯 ACTUAL MODEL OUTPUT — verify mean pooling against model.encode()
encoded = tokenizer(text, return_tensors="pt")          # tokens -> ids + mask (Section 4)
print("input ids      :", encoded["input_ids"].tolist())
print("attention mask :", encoded["attention_mask"].tolist(), "  <- 1 = real token")

with torch.no_grad():
    outputs = model[0].auto_model(**encoded)            # run the REAL Transformer
token_vecs = outputs.last_hidden_state                   # (1, T, 384) contextual vectors
print("\n🎯 last_hidden_state shape:", tuple(token_vecs.shape),
      "= (batch=1, tokens=T, dim=384)")
print("   first 3 dims of the 'female' token vector:", np.round(token_vecs[0, 1, :3].numpy(), 4))

# ---- mean pooling, by hand ----
mask      = encoded["attention_mask"].unsqueeze(-1).float()   # (1, T, 1)
summed    = (token_vecs * mask).sum(dim=1)                    # (1, 384) sum of real tokens
counts    = mask.sum(dim=1)                                   # (1, 1)   number of real tokens
mean_vec  = summed / counts                                   # (1, 384) the average
manual    = mean_vec / mean_vec.norm(dim=1, keepdim=True)     # L2-normalize, like encode()

real = model.encode(text, normalize_embeddings=True)          # what the library returns
print("\n🎯 hand-made mean-pooled vector == model.encode() output ?",
      np.allclose(manual[0].numpy(), real, atol=1e-5))
print("   max abs difference:", float(np.abs(manual[0].numpy() - real).max()))


In [ ]:

# 🎯 Bonus: why mean pooling (and not just the [CLS] token)?
cls_vec = token_vecs[0, 0]                                   # vector of the [CLS] token
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(manual[0].numpy().reshape(1, -1), cls_vec.numpy().reshape(1, -1))[0, 0]
print("🎯 ACTUAL OUTPUT")
print("  cosine similarity between mean-pooled vector and raw [CLS] vector:", round(float(sim), 4))
print("  -> MiniLM was TRAINED to use the mean of all tokens, not the [CLS] token,")
print("     so that is what sentence-transformers uses as pooling.")



## 💡 What we just learned
- Pooling turns a variable-length sequence of token vectors into one fixed-size vector.
- For MiniLM: **mean over real tokens + L2 normalization**. We reproduced `model.encode()` by hand —
  so we now know exactly what the library does.
- Pooling choice matters: mean / CLS / max pooling differ, and each model is trained for its own.

## 🎤 Interview checkpoint
**Q: Why do we need pooling in sentence embedding models?**
**A:** The Transformer produces a vector per token; retrieval needs one vector per chunk.
Pooling (e.g., mean over token vectors) aggregates them into a single fixed-size embedding.



---

# 📌 SECTION 11 — VECTOR SPACE & PCA VISUALIZATION

## 📌 Simple definition
A vector is a **point / direction in mathematical space**. Semantically similar texts live close
together in this space. Our real vectors have 384 dimensions — impossible to draw. So we **project**
them to 2D with **PCA** *only for visualization*.

> ⚠️ The 2D plot is a **projection** of the high-dimensional space. Distances in 2D are only
> approximate; the true geometry lives in 384 dimensions.

## 💻 Embed our 5 real policy chunks and project them


In [ ]:

# The 5 topical chunks of our leave policy (real text from the PDF)
topic_chunks = [
    "Employees receive 18 days of annual leave per year.",
    "Employees receive 10 days of paid sick leave per year.",
    "Female employees receive 26 weeks of paid maternity leave.",
    "Employees receive 7 days of paid paternity leave.",
    "Employees may work remotely up to 2 days per week with manager approval.",
]

emb5 = model.encode(topic_chunks, normalize_embeddings=True)   # (5, 384) REAL embeddings
print("🎯 ACTUAL OUTPUT — embeddings shape:", emb5.shape, "(5 chunks x 384 dims)")

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
xy = pca.fit_transform(emb5)                                   # (5, 2) projection

labels = ["Annual", "Sick", "Maternity", "Paternity", "WFH"]
fig, ax = plt.subplots(figsize=(8, 6))
for (x, y), lab, vec in zip(xy, labels, emb5):
    ax.scatter(x, y, s=120)
    ax.annotate(lab, (x, y), textcoords="offset points", xytext=(8, 8), fontsize=12)
    # also show the 3 nearest neighbours in TRUE 384-d space for one point of reference
ax.set_title("PCA projection of REAL 384-d chunk embeddings\n(2D projection for visualization ONLY)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("🎯 variance kept by 2 PCs:", round(float(pca.explained_variance_ratio_.sum()), 4),
      " (of the original 384-d information)")


In [ ]:

# 🎯 True-space check: nearest neighbour of each chunk in the REAL 384-d space
sim_matrix = cosine_similarity(emb5)
for i, lab in enumerate(labels):
    order = np.argsort(-sim_matrix[i])
    nn = [labels[j] for j in order if j != i][:2]
    print(f"  {lab:<10} closest to: {nn}   (cosine sims {[round(sim_matrix[i, j], 3) for j in order if j != i][:2]})")



## 💡 What we just learned
- Similar meaning → nearby direction in embedding space (Maternity is close to Paternity, far from WFH).
- PCA is only a 2D **view**; retrieval decisions are made in the full 384-d space.

## 🎤 Interview checkpoint
**Q: How do you visualize high-dimensional embeddings?**
**A:** Project them with a dimensionality-reduction technique like PCA or t-SNE/UMAP.
The plot is an approximation — similarity is computed in the original space, not the projection.



---

# 📌 SECTION 12 — QUERY EMBEDDING

## 📌 Simple definition
The user's question must live in the **same vector space** as the chunks to be comparable.
So we embed the query with the **exact same model**.

```
QUERY  "How many weeks of maternity leave are available for a female worker?"
   ↓  SAME embedding model (never mix models for queries and documents!)
query vector  (384,)
```


In [ ]:

QUERY = "How many weeks of maternity leave are available for a female worker?"
q_emb = model.encode(QUERY, normalize_embeddings=True)

print("🎯 ACTUAL OUTPUT")
print("  query            :", QUERY)
print("  query emb shape  :", q_emb.shape)
print("  first 10 values  :", np.round(q_emb[:10], 6))
print("  L2 norm          :", round(float(np.linalg.norm(q_emb)), 6))



## 💡 What we just learned
- Query and chunks must be embedded with the **same model** — otherwise the spaces don't match and
  similarity is meaningless.
- The query is embedded at query time (it is new), while chunks were embedded at ingestion time.

## 🎤 Interview checkpoint
**Q: Can you embed queries and documents with different models?**
**A:** No (in standard bi-encoder RAG). Both must map into the same learned vector space for
distance/similarity to be meaningful.



---

# 📌 SECTION 13 — COSINE SIMILARITY (the retrieval score)

## 📌 Simple definition
Cosine similarity measures the **directional similarity** between two vectors:

```
cosine_similarity(q, c) = (q · c) / (||q|| · ||c||)

q · c   = dot product   = q1·c1 + q2·c2 + ...      (sum of element-wise products)
||q||   = norm of q     = sqrt(q1² + q2² + ...)
```

- Result is between **−1 and 1**; 1 = same direction, 0 = orthogonal, −1 = opposite.
- It ignores magnitude — only **angle/direction** matters (which is why we normalized earlier).

## 🔢 Tiny REAL numerical example (2D, real arithmetic — hand-calculable)
`q = [3, 4]`, `c = [1, 2]`


In [ ]:

# 🔢 REAL arithmetic on a tiny 2D example (the math is real; the vectors are just small)
q = np.array([3.0, 4.0])
c = np.array([1.0, 2.0])

dot   = q @ c                                   # step 1
nq    = np.linalg.norm(q)                       # step 2
nc    = np.linalg.norm(c)                       # step 3
sim   = dot / (nq * nc)                         # step 4

print("step 1  dot product q·c  =", dot,   "   (3*1 + 4*2)")
print("step 2  ||q||            =", nq,    "   sqrt(3² + 4²)")
print("step 3  ||c||            =", round(nc, 4), "   sqrt(1² + 2²)")
print("step 4  cosine           =", round(dot, 4), "/", round(nq, 4), "/", round(nc, 4),
      "=", round(sim, 4))

# Visual: the angle between the two vectors
fig, ax = plt.subplots(figsize=(5, 5))
ax.quiver(0, 0, q[0], q[1], angles="xy", scale_units="xy", scale=1, color="tab:blue", label="q = [3,4]")
ax.quiver(0, 0, c[0], c[1], angles="xy", scale_units="xy", scale=1, color="tab:red", label="c = [1,2]")
angle = np.degrees(np.arccos(np.clip(sim, -1, 1)))
ax.set_title(f"angle between q and c = {angle:.1f}°  ->  cosine = {sim:.3f}")
ax.set_xlim(0, 5); ax.set_ylim(0, 5); ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax.legend(); plt.tight_layout(); plt.show()



## 🎯 Now the same formula on our REAL 384-d vectors
`sklearn.metrics.pairwise.cosine_similarity` implements exactly that formula, on full-size vectors.


In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(q_emb.reshape(1, -1), emb5)[0]     # (1,384) x (5,384)^T -> (5,)

table = pd.DataFrame({
    "Chunk": labels,
    "Text": topic_chunks,
    "Cosine similarity": np.round(scores, 4),
}).sort_values("Cosine similarity", ascending=False).reset_index(drop=True)
table.index = table.index + 1
print("🎯 ACTUAL OUTPUT — REAL cosine similarities, sorted descending\n")
print(table.to_string())

fig, ax = plt.subplots(figsize=(8, 3.5))
order = np.argsort(scores)
ax.barh([labels[i] for i in order], scores[order], color="tab:blue", alpha=0.7)
ax.set_xlabel("cosine similarity vs query")
ax.set_title("REAL similarities: query vs the 5 chunks")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()



## 💡 What we just learned
- Retrieval score = cosine similarity between query vector and each chunk vector.
- The maternity chunk wins (as it should!) — and these are the model's **actual** numbers, not
  hand-picked ones.

## 🎤 Interview checkpoint
**Q: Why cosine similarity instead of Euclidean distance?**
**A:** Cosine measures direction only, so it is robust to text length and magnitude differences;
it is the standard score for comparing normalized embeddings.



---

# 📌 SECTION 14 — VECTOR DATABASE (ChromaDB)

## 📌 Simple definition
A **vector database** stores `(vector, text, metadata)` triples and answers
*"which stored vectors are nearest to this query vector?"* efficiently (approximate nearest
neighbour search — ANN — instead of comparing against every chunk one by one).

## 🧠 Intuition — a crucial distinction for interviews
```
EMBEDDING MODEL   -> understands LANGUAGE (turns text into numbers)
VECTOR DATABASE   -> understands NOTHING about English!
```
The DB is a dumb-but-fast warehouse of numbers. It stores vectors, indexes them, and searches
them. The *meaning* is entirely created by the embedding model. If you change the model, you must
**re-embed everything** — the old numbers belong to the old model's space.

## 🔢 What we store
```
VECTOR DB  (collection: "leave_policy")
┌──────────┬──────────────┬──────────────────────────────────┐
│ chunk id │ vector       │ document text + metadata         │
├──────────┼──────────────┼──────────────────────────────────┤
│ chunk_00 │ [0.03, ...]  │ "Employees receive 18 days..."   │ {source, page} │
│ chunk_01 │ [-0.01, ...] │ "Employees receive 10 days..."   │ {source, page} │
│ ...      │   (384 dims) │ ...                              │ ...            │
└──────────┴──────────────┴──────────────────────────────────┘
```


In [ ]:

import chromadb

client = chromadb.PersistentClient(path="./chroma_db")   # saved on disk inside the Colab VM
collection = client.get_or_create_collection(
    name="leave_policy",
    metadata={"hnsw:space": "cosine"},   # compare vectors with COSINE distance
)

# Idempotency: if you re-run this cell, replace the old data instead of duplicating it.
existing = collection.get()["ids"]
if existing:
    collection.delete(ids=existing)
    print(f"cleared previous run ({len(existing)} chunks)")

# ---- INGESTION: embed every chunk ONCE, then store ----
chunk_embs = model.encode(chunks, normalize_embeddings=True).tolist()   # REAL embeddings
ids        = [f"chunk_{i:02d}" for i in range(len(chunks))]
metadatas  = [{"source": "leave_policy.pdf", "page": 1, "chunk_index": i}
              for i in range(len(chunks))]

collection.add(
    ids=ids,
    embeddings=chunk_embs,
    documents=chunks,
    metadatas=metadatas,
)

print(f"✅ stored {collection.count()} chunks in the vector DB")
print("\n🎯 ACTUAL OUTPUT — peek into the DB (first 2 entries):")
for d, m in zip(collection.peek(limit=2)["documents"], collection.peek(limit=2)["metadatas"]):
    print("  text     :", d[:80], "...")
    print("  metadata :", m)



## 💡 What we just learned
- The DB stores **vectors + payload (text + metadata)**. Retrieval returns the payload, not just numbers.
- Embedding happens **once per chunk** (ingestion time); the DB only ever sees numbers.
- Our collection is configured with **cosine** space, so the DB's `distance = 1 − cosine similarity`.

## 🎤 Interview checkpoint
**Q: Why use a vector database instead of plain NumPy?**
**A:** For scale: ANN indexes (like HNSW) make nearest-neighbour search fast on millions of
vectors, with persistence, metadata filtering, and CRUD — while a brute-force NumPy comparison is O(N) per query.



---

# 📌 SECTION 15 — TOP-K RETRIEVAL

## 📌 Simple definition
**Top-K retrieval**: embed the query, ask the vector DB for the **K most similar chunks**, and
return them ranked by similarity. These are **candidates**, not yet the final answer.

```
QUERY  "How many weeks of maternity leave are available for a female worker?"
   ↓  embed with the same model (Section 12)
query vector
   ↓  vector DB search (cosine)
top-K chunks, ranked
```


In [ ]:

K = 3
results = collection.query(
    query_embeddings=[q_emb.tolist()],     # the real query vector from Section 12
    n_results=K,
    include=["documents", "metadatas", "distances"],
)

print(f"🎯 ACTUAL OUTPUT — top-{K} retrieved chunks\n")
rows = []
for rank, (doc, meta, dist) in enumerate(zip(results["documents"][0],
                                             results["metadatas"][0],
                                             results["distances"][0]), start=1):
    sim = 1.0 - dist        # cosine space: similarity = 1 - cosine distance
    rows.append({"Rank": rank,
                 "Similarity": round(sim, 4),
                 "Chunk": doc[:70],
                 "Source": meta["source"],
                 "Page": meta["page"]})
print(pd.DataFrame(rows).to_string(index=False))



## 💡 What we just learned
- The DB returns ranked candidates **with their text and metadata** — ready to become context.
- Top-K means "give me the K best candidates". K is a trade-off: too small → missing evidence;
  too large → noise (we test both in Section 21).

## 🎤 Interview checkpoint
**Q: How does retrieval work in a RAG pipeline?**
**A:** The query is embedded with the same model as the documents; the vector DB finds the K
chunks with highest cosine similarity to the query vector and returns them with their metadata.



---

# 📌 SECTION 16 — RE-RANKING WITH A CROSS-ENCODER

## 📌 Simple definition
Retrieval compares **two pre-computed vectors** (fast, approximate). Re-ranking lets the query and
each candidate **interact directly inside a Transformer** (slower, but far more precise).

## 🔢 Bi-encoder vs cross-encoder
```
BI-ENCODER (retrieval)                    CROSS-ENCODER (re-ranking)
query  → vector            query + chunk  →  [q] [SEP] [c]   (one input!)
chunk  → vector                  ↓ Transformer (full attention between q and c)
score = cosine(q_vec, c_vec)     ↓ classifier head
                                 relevance score (logit)
```
- Bi-encoder: independent vectors, comparable in O(1) after indexing → scales to millions.
- Cross-encoder: exact word-by-word interaction between query and chunk → more accurate, but must
  re-run the model for every (query, chunk) pair → only affordable on a small candidate set.

## 💻 Real cross-encoder
Model: `cross-encoder/ms-marco-MiniLM-L-6-v2`, trained on MS MARCO relevance judgments.


In [ ]:

from sentence_transformers import CrossEncoder

CROSS_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
try:
    cross = CrossEncoder(CROSS_MODEL)          # ~90 MB download
except Exception as e:
    print(f"WARNING: Could not load {CROSS_MODEL}: {e}")
    print("Re-ranking will be skipped in downstream cells.")
    cross = None

# Take MORE candidates than we finally need: retrieve 5, keep the best 3 after re-ranking.
n_candidates = 5
res5 = collection.query(query_embeddings=[q_emb.tolist()], n_results=n_candidates,
                        include=["documents", "metadatas", "distances"])
cand_docs  = res5["documents"][0]
cand_metas = res5["metadatas"][0]
cand_dists = res5["distances"][0]

# Cross-encoder scores each (query, chunk) PAIR — the model sees them together.
if cross is not None:
    ce_scores = cross.predict([[QUERY, doc] for doc in cand_docs])     # REAL scores
    print("🎯 ACTUAL OUTPUT — cross-encoder scores:", np.round(ce_scores, 4))
else:
    ce_scores = np.zeros(len(cand_docs))
    print("⚠️ Cross-encoder not loaded — using cosine scores as fallback")


In [ ]:

# 📊 Before (cosine) vs after (cross-encoder) ranking
ce_order = np.argsort(-ce_scores)
n_cands = len(cand_docs)     # may be fewer than requested if the corpus is tiny

before = pd.DataFrame({
    "Bi-encoder rank": range(1, n_cands + 1),
    "Chunk": [c[:60] for c in cand_docs],
    "Cosine sim": [round(1 - d, 4) for d in cand_dists],
})
after = pd.DataFrame({
    "Cross-encoder rank": range(1, n_cands + 1),
    "Chunk": [cand_docs[i][:60] for i in ce_order],
    "Cross-encoder score": [round(float(ce_scores[i]), 4) for i in ce_order],
})
print("🎯 BEFORE re-ranking (bi-encoder / cosine)\n"); print(before.to_string(index=False))
print("\n🎯 AFTER re-ranking (cross-encoder)\n"); print(after.to_string(index=False))
print("\n(re-ranking keeps the SAME candidates — it only re-orders them by joint relevance)")



## 💡 What we just learned
- **Two-stage retrieval**: cheap bi-encoder recall on everything → expensive cross-encoder
  precision on the top few.
- The re-ranker sees `query + chunk` together, so it catches paraphrases and word-overlap nuances
  that vector comparison can miss.

## 🎤 Interview checkpoint
**Q: Why re-rank instead of using a cross-encoder on everything?**
**A:** Cross-encoders are too slow to score millions of chunks per query. The fast bi-encoder
first narrows to a small candidate set; the cross-encoder then re-orders only those candidates
for precise relevance.



---

# 📌 SECTION 17 — BUILD THE RAG PROMPT (manually — no LangChain)

## 📌 Simple definition
The prompt is simply a **string** with three parts:
1. **System instruction** — answer only from the context, admit when it's not there.
2. **Context** — the re-ranked chunks (with source labels).
3. **Question** — the user's query.

Nothing magical: retrieval + string formatting.

```
SYSTEM:
You are an HR assistant. Answer using ONLY the supplied context.
If the answer is not in the context, say you don't know.

CONTEXT:
[1] Female employees receive 26 weeks of paid maternity leave.  (leave_policy.pdf, p.1)
[2] ...

QUESTION:
How many weeks of maternity leave are available for a female worker?

ANSWER:
```


In [ ]:

def build_prompt(query, docs_with_meta, max_chunks=3):
    """docs_with_meta: list of (text, metadata) in FINAL relevance order (after re-ranking)."""
    selected = docs_with_meta[:max_chunks]
    context = "\n\n".join(
        f"[{i}] {text}\n    (source: {meta['source']}, page {meta['page']})"
        for i, (text, meta) in enumerate(selected, start=1)
    )
    return f"""You are an HR assistant. Answer using ONLY the supplied context.
If the answer is not present in the context, say you don't know.
Do not use outside knowledge.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

# Re-ranked candidates, in cross-encoder order, with their metadata:
final_candidates = [(cand_docs[i], cand_metas[i]) for i in ce_order]

prompt = build_prompt(QUERY, final_candidates, max_chunks=3)
print("🎯 ACTUAL OUTPUT — the exact prompt we send to the LLM:")
print("=" * 60)
print(prompt)
print("=" * 60)

n_prompt_tokens = len(tokenizer.tokenize(prompt))
print(f"\nprompt length: {len(prompt)} characters, ~{n_prompt_tokens} tokens (context + question + instructions)")



## 💡 What we just learned
- The RAG prompt is plain text: instructions + retrieved context + question.
- Keeping sources inside the context lets the LLM (and you) cite them.
- Prompt length is dominated by the context — another reason top-K should not be huge.

## 🎤 Interview checkpoint
**Q: How do you prevent an LLM from hallucinating in RAG?**
**A:** Constrain it with a system instruction to answer only from the supplied context and to say
"don't know" otherwise, and provide verified retrieved context in the prompt.



---

# 📌 SECTION 18 — THE FINAL LLM ANSWER

## 📌 Simple definition
The prompt goes to an LLM, which generates the grounded answer. We then attach the **source**
of the top retrieved chunk for citation.

**Two modes in this notebook:**
- 🔑 **Real LLM mode** — pick an OpenAI-compatible provider (presets: **OpenCode Zen**, OpenAI,
  Google Gemini, Groq — or any custom base URL), paste your API key, and choose the model.
  The key is read with `getpass` and is **never saved** in the notebook.
- 🧪 **DEMO mode** — if you skip the key, the pipeline still completes using a clearly-labelled
  rule-based placeholder. ⚠️ **That answer is NOT an LLM answer.**

> 💡 Any service that exposes an OpenAI-style `/chat/completions` endpoint works here.
> The notebook will try to **list the models your key can access** so you pick a real one.

```
Question → query embedding → vector search → top-K → re-ranking → best context
         → prompt → LLM → answer (+ source citation)
```


In [ ]:

import requests

def call_openai_compatible(prompt, api_key, base_url="https://api.openai.com/v1",
                           model_name="gpt-4o-mini", timeout=60, max_retries=2):
    # Minimal OpenAI-compatible chat completion via raw REST (no SDK needed).
    # Works for OpenCode Zen, OpenAI, Gemini's OpenAI-compat endpoint, Groq, Ollama...
    # Retries on transient errors (429, 500, 502, 503, 504).
    import time
    url = f"{base_url}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}"}
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system",
             "content": "Answer using only the supplied context. Say you don't know if absent."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }
    last_err = None
    for attempt in range(max_retries + 1):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", 5))
                print(f"  (rate-limited, waiting {retry_after}s before retry {attempt+1}/{max_retries})")
                time.sleep(retry_after)
                continue
            if resp.status_code >= 500:
                print(f"  (server error {resp.status_code}, retry {attempt+1}/{max_retries})")
                time.sleep(2 ** attempt)
                continue
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"]
        except requests.exceptions.ConnectionError as e:
            last_err = e
            print(f"  (connection error: {e}, retry {attempt+1}/{max_retries})")
            time.sleep(2 ** attempt)
        except requests.exceptions.Timeout as e:
            last_err = e
            print(f"  (timeout after {timeout}s, retry {attempt+1}/{max_retries})")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"LLM call failed after {max_retries+1} attempts: {last_err}")

def list_models(base_url, api_key):
    """Try GET {base_url}/models and return model ids, or None if unsupported."""
    try:
        r = requests.get(f"{base_url}/models",
                         headers={"Authorization": f"Bearer {api_key}"}, timeout=30)
        r.raise_for_status()
        data = r.json().get("data", [])
        return [m["id"] for m in data if isinstance(m, dict) and "id" in m]
    except Exception:
        return None

# ---- provider presets (OpenAI-compatible base URLs) ----
PROVIDERS = {
    "1": ("OpenCode Go",   "https://opencode.ai/zen/go/v1",                         "deepseek-v4-flash"),
    "2": ("OpenCode Zen",  "https://opencode.ai/zen/v1",                              "deepseek-v4-flash"),
    "3": ("OpenAI",        "https://api.openai.com/v1",                               "gpt-4o-mini"),
    "4": ("Google Gemini", "https://generativelanguage.googleapis.com/v1beta/openai", "gemini-2.0-flash"),
    "5": ("Groq",          "https://api.groq.com/openai/v1",                          "llama-3.3-70b-versatile"),
    "6": ("Custom base URL", None, None),
}
print("Choose your LLM provider:")
for k, (name, base, _m) in PROVIDERS.items():
    print(f"  [{k}] {name}" + (f"   ({base})" if base else "   (you type the base URL)"))

choice    = input("Provider number (default 1 = OpenCode Go): ").strip() or "1"
pname, base_url, default_model = PROVIDERS[choice]
if base_url is None:
    base_url      = input("Base URL (e.g. https://host/v1): ").strip().rstrip("/")
    default_model = input("Model name: ").strip()

# ---- get the key securely (or skip -> DEMO mode) ----
api_key = os.environ.get("LLM_API_KEY", "").strip() or getpass.getpass(
    f"API key for {pname} (press Enter to skip -> DEMO mode): "
).strip()

# ---- pick the model: list what the key can access, then choose ----
llm_model = default_model
if api_key:
    available = list_models(base_url, api_key)
    if available:
        print(f"🎯 {len(available)} model(s) available via your key:")
        for m in available[:15]:
            print("   -", m)
        pick = input(f"Model name (Enter = default {default_model!r}): ").strip()
        if pick:
            llm_model = pick
    else:
        print(f"(could not list models from {base_url}/models — some providers hide this endpoint)")
        pick = input(f"Model name (Enter = default {default_model!r}): ").strip()
        if pick:
            llm_model = pick
    print(f"→ using {pname} @ {base_url} | model: {llm_model}")

top_doc, top_meta = final_candidates[0]      # best chunk after re-ranking

if api_key:
    try:
        answer = call_openai_compatible(prompt, api_key, base_url=base_url, model_name=llm_model)
        mode = f"🤖 REAL LLM ANSWER ({pname} / {llm_model})"
    except Exception as e:
        print(f"⚠️ LLM call failed ({type(e).__name__}: {e}).")
        print("   Check base URL / model name / key validity. Falling back to DEMO mode.")
        api_key = ""                          # fall through to demo

if not api_key:
    mode = "🧪 DEMO MODE — NOT an actual LLM answer (rule-based placeholder)"
    answer = (f"The retrieved context states: '{top_doc}' "
              f"(this echo is a placeholder — connect an API key to get a real LLM answer).")

print("=" * 70)
print(mode)
print("=" * 70)
print(answer)
print("-" * 70)
print(f"📚 Source: {top_meta['source']} — page {top_meta['page']}")
print("=" * 70)



## 💡 What we just learned
- Generation is the **last** step — retrieval and re-ranking did the heavy lifting.
- Grounding = the answer comes from retrieved context; citations come from the metadata we
  preserved all the way from Section 2.
- In demo mode the pipeline is complete but the "answer" is a labelled placeholder — never
  present demo output as a real LLM response.

## 🎤 Interview checkpoint
**Q: How does RAG reduce hallucination?**
**A:** By grounding generation in retrieved evidence: the LLM answers from supplied context
instead of relying on parametric memory, and the system can cite the source documents.



---

# 📌 SECTION 19 — VISUAL RAG DASHBOARD

One picture of the whole pipeline, annotated with the **actual data** from this run.


In [ ]:

# 📊 Draw the full pipeline as a flowchart, annotated with REAL values from this session.
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 16))
ax.set_xlim(0, 10); ax.set_ylim(0, 18); ax.axis("off")

first_chunk = chunks[0]
stages = [
    ("📄 PDF",                  f"leave_policy.pdf — {os.path.getsize(pdf_path)} bytes, 1 page"),
    ("📖 Extracted Text",       f"{len(full_text)} chars, 1 page object with metadata"),
    ("✂️ Chunks",              f"{len(chunks)} chunks ({CHUNK_SIZE} words, overlap {CHUNK_OVERLAP})"),
    ("🔤 Tokens",               f"{len(tokenizer.tokenize(text))} tokens for: '{text[:38]}...'"),
    ("🔢 Token IDs",            str(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))),
    ("📐 Embedding",            f"{emb.shape} = (384,) real MiniLM vector, norm={np.linalg.norm(emb):.2f}"),
    ("🗄️ Vector DB",           f"ChromaDB collection 'leave_policy' — {collection.count()} stored vectors"),
    ("🔍 Query Search",         f"query -> (384,) vector, cosine vs all {collection.count()} chunks"),
    ("🏆 Re-ranking",           f"cross-encoder re-scored top {len(cand_docs)} candidates"),
    ("📚 Context",              f"top chunk: '{top_doc[:45]}...'  (page {top_meta['page']})"),
    ("🤖 LLM",                  "prompt = system + context + question"),
    ("✅ Answer",               f"'...{top_doc[:45]}...' + citation: {top_meta['source']} p.{top_meta['page']}"),
]

y = 17.2
for name, detail in stages:
    ax.add_patch(FancyBboxPatch((1.5, y - 0.55), 7.0, 1.1,
                 boxstyle="round,pad=0.08", facecolor="#eef3ff", edgecolor="#4477cc", lw=1.5))
    ax.text(5.0, y + 0.16, name, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(5.0, y - 0.26, detail, ha="center", va="center", fontsize=9, color="#333333",
            wrap=True)
    if y < 17.2:
        ax.annotate("", xy=(5.0, y + 0.62), xytext=(5.0, y + 1.45),
                    arrowprops=dict(arrowstyle="-|>", color="#999999", lw=1.5))
    y -= 1.45

ax.set_title("Mini RAG Learning Lab — live pipeline snapshot", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()



---

# 📌 SECTION 20 — INTERACTIVE LEARNING

Play with the pieces yourself.


In [ ]:

from ipywidgets import interact, IntSlider

# 🎛️ Pick any chunk -> see its tokens, IDs, embedding shape and first 10 real values
@interact(chunk_idx=IntSlider(min=0, max=len(chunks) - 1, step=1, value=3,
                              description="chunk:"))
def show_chunk(chunk_idx):
    c = chunks[chunk_idx]
    print("TEXT :", c)
    print("TOKENS + IDs:")
    for t, i in zip(tokenizer.tokenize(c), tokenizer.convert_tokens_to_ids(tokenizer.tokenize(c))):
        print(f"   {t!r:<18} -> {i}")
    vec = model.encode(c, normalize_embeddings=True)
    print(f"\nEMBEDDING: shape {vec.shape}, first 10 values:")
    print("  ", np.round(vec[:10], 5))


In [ ]:

# 🎛️ Choose K and (optionally) re-ranking -> see retrieved results live
from ipywidgets import interact, IntSlider, Checkbox

def run_search(k=3, use_rerank=True):
    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=k,
                           include=["documents", "metadatas", "distances"])
    docs, metas, dists = res["documents"][0], res["metadatas"][0], res["distances"][0]
    order = list(range(len(docs)))
    if use_rerank and len(docs) > 1 and cross is not None:
        s = cross.predict([[QUERY, d] for d in docs])
        order = np.argsort(-s)
        print(f"(cross-encoder scores: {np.round(s, 3)})")
    elif use_rerank and cross is None:
        print("(cross-encoder not available — showing cosine order)")
    for rank, i in enumerate(order, start=1):
        print(f"{rank}. sim={1 - dists[i]:.4f} | {docs[i][:55]}... | {metas[i]['source']} p.{metas[i]['page']}")

interact(run_search, k=IntSlider(min=1, max=min(5, len(chunks)), value=3), use_rerank=True)



---

# 📌 SECTION 20B — ADVANCED CHUNKING STRATEGIES

The naive word-based chunker is easy to understand, but real systems use more
sophisticated strategies. Let's compare three approaches on our document.

**Strategy 1 — Naive (fixed word count):** Split every N words. Simple but can split sentences.

**Strategy 2 — Sentence-aware:** Split on sentence boundaries (`.`, `!`, `?`). No chunk cuts a sentence in half.

**Strategy 3 — Recursive:** Try paragraph breaks first, then sentences, then words. Respects document structure.


In [ ]:

import re

def chunk_by_sentence(text, max_words=25):
    # Split on sentence boundaries, merge short sentences into chunks.
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, word_count = [], [], 0
    for s in sentences:
        sw = len(s.split())
        if word_count + sw > max_words and current:
            chunks.append(' '.join(current))
            current, word_count = [], 0
        current.append(s)
        word_count += sw
    if current:
        chunks.append(' '.join(current))
    return chunks

def chunk_recursive(text, max_words=25, overlap_words=4):
    # Recursive splitting: paragraph -> sentence -> word boundaries.
    if len(text.split()) <= max_words:
        return [text.strip()]
    # try paragraph split first
    paragraphs = text.split('\n\n')
    if len(paragraphs) > 1:
        result = []
        for p in paragraphs:
            if p.strip():
                result.extend(chunk_recursive(p.strip(), max_words, overlap_words))
        return result
    # fall back to sentence split
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) > 1:
        result, current, wc = [], [], 0
        for s in sentences:
            sw = len(s.split())
            if wc + sw > max_words and current:
                result.append(' '.join(current))
                current, wc = [], 0
            current.append(s)
            wc += sw
        if current:
            result.append(' '.join(current))
        return result
    # last resort: word-based with overlap
    return chunk_text(text, chunk_size_words=max_words, overlap_words=overlap_words)

# Compare all three strategies
naive_chunks = chunk_text(full_text, chunk_size_words=20, overlap_words=4)
sentence_chunks = chunk_by_sentence(full_text, max_words=25)
recursive_chunks = chunk_recursive(full_text, max_words=25, overlap_words=4)

print(f"{'Strategy':<20} | {'Chunks':>6} | {'Avg words/chunk':>15}")
print('-' * 50)
print(f"{'Naive (20 words)':<20} | {len(naive_chunks):>6} | {np.mean([len(c.split()) for c in naive_chunks]):>15.1f}")
print(f"{'Sentence-aware':<20} | {len(sentence_chunks):>6} | {np.mean([len(c.split()) for c in sentence_chunks]):>15.1f}")
print(f"{'Recursive':<20} | {len(recursive_chunks):>6} | {np.mean([len(c.split()) for c in recursive_chunks]):>15.1f}")

print()
for name, cs in [('Sentence-aware', sentence_chunks), ('Recursive', recursive_chunks)]:
    print(f'--- {name} chunks ---')
    for i, c in enumerate(cs):
        print(f'  [{i}] ({len(c.split())} words) {c[:70]}...')
    print()


In [ ]:

# Compare retrieval quality across chunking strategies
q_chunks_naive = model.encode(naive_chunks, normalize_embeddings=True)
q_chunks_sentence = model.encode(sentence_chunks, normalize_embeddings=True)
q_chunks_recursive = model.encode(recursive_chunks, normalize_embeddings=True)

sim_naive = cosine_similarity(q_emb.reshape(1, -1), q_chunks_naive)[0].max()
sim_sentence = cosine_similarity(q_emb.reshape(1, -1), q_chunks_sentence)[0].max()
sim_recursive = cosine_similarity(q_emb.reshape(1, -1), q_chunks_recursive)[0].max()

print('Best cosine similarity for the maternity query:')
print(f'  Naive (20 words)   : {sim_naive:.4f}')
print(f'  Sentence-aware     : {sim_sentence:.4f}')
print(f'  Recursive          : {sim_recursive:.4f}')
print()
print('All strategies find the relevant chunk for this simple document.')
print('For longer, more complex documents, sentence-aware and recursive')
print('chunking tend to preserve more context per chunk.')



## 💡 Key points
- **Naive chunking** is simple but can split sentences and paragraphs.
- **Sentence-aware** respects sentence boundaries — no chunk cuts a sentence in half.
- **Recursive** tries paragraph → sentence → word boundaries, preserving structure.
- The best strategy depends on the document: structured docs benefit from recursive;
  dialogue or informal text may work fine with naive.

## 🎤 Interview checkpoint
**Q: What chunking strategies exist beyond fixed-size windows?**
**A:** Sentence-aware (split on sentence boundaries), recursive (paragraph → sentence → word
fallback), and semantic (split where consecutive sentence embeddings diverge). Each trades
simplicity for better context preservation.



---

# 📌 SECTION 21 — RAG FAILURE EXPERIMENTS

RAG quality fails in predictable ways. Here are six live experiments — run them and read the
actual numbers.

### 🧪 Experiment 1 — Badly sized chunks
Tiny chunks (3 words) break the "one idea per chunk" rule: the answer gets scattered.


In [ ]:

tiny_chunks = chunk_text(full_text, chunk_size_words=3, overlap_words=0)
tiny_embs = model.encode(tiny_chunks, normalize_embeddings=True)
tscores = cosine_similarity(q_emb.reshape(1, -1), tiny_embs)[0]
best = int(np.argmax(tscores))

maternity_chunk = next(c for c in chunks if "maternity" in c.lower())   # the real normal chunk
maternity_sim = float(cosine_similarity(
    q_emb.reshape(1, -1),
    model.encode(maternity_chunk, normalize_embeddings=True).reshape(1, -1))[0, 0])

print(f"🎯 ACTUAL OUTPUT — {len(tiny_chunks)} tiny chunks created")
print(f"  best tiny chunk: '{tiny_chunks[best]}'  (sim={tscores[best]:.3f})")
print(f"  vs normal chunk: '{maternity_chunk[:60]}'  (sim={maternity_sim:.3f})")
print("\n💡 The tiny chunk contains only a fragment ('26 weeks of') — it cannot answer the question")
print("   by itself. Context was chopped into pieces too small to be useful.")



### 🧪 Experiment 2 — No overlap vs overlap at a phrase boundary
We place a cut **inside** the key phrase and compare retrieval with and without overlap.


In [ ]:

boundary_text = ("Female employees receive 26 weeks of paid maternity leave and "
                 "7 days of paid paternity leave per year.")
q2 = "How many weeks of maternity leave are available?"

no_ov  = chunk_text(boundary_text, chunk_size_words=7, overlap_words=0)   # cut at word 7
with_ov = chunk_text(boundary_text, chunk_size_words=7, overlap_words=3)

print("🎯 no overlap chunks:")
for c in no_ov: print("   ", repr(c))
print("\n🎯 with overlap chunks:")
for c in with_ov: print("   ", repr(c))

q2_emb = model.encode(q2, normalize_embeddings=True)
print("\n🎯 ACTUAL OUTPUT — best similarity for query 'How many weeks of maternity leave...'")
print(f"  no overlap : max sim = {cosine_similarity(q2_emb.reshape(1,-1), model.encode(no_ov)).max():.3f}")
print(f"  with overlap: max sim = {cosine_similarity(q2_emb.reshape(1,-1), model.encode(with_ov)).max():.3f}")
print("\n💡 Without overlap, '26 weeks' and 'maternity leave' were split across two chunks,")
print("   so NO chunk contains the full fact. Overlap keeps the phrase together.")



### 🧪 Experiment 3 — Wrong retrieval (question the document cannot answer)
A good RAG system must be able to say "I don't know".


In [ ]:

q_bad = "How many days of vacation can my pet iguana take?"
q_bad_emb = model.encode(q_bad, normalize_embeddings=True)
res_bad = collection.query(query_embeddings=[q_bad_emb.tolist()], n_results=3,
                           include=["documents", "distances"])
for d, dist in zip(res_bad["documents"][0], res_bad["distances"][0]):
    print(f"  sim={1 - dist:.3f} | {d[:60]}")
print("\n💡 All similarities are LOW (≈0.3 or less). A threshold or an honest prompt")
print("   ('say you don't know') prevents the LLM from inventing an answer.")



### 🧪 Experiment 4 — Top-K too small
The question needs TWO facts; K=1 can only return one.


In [ ]:

q_multi = "How many annual leave days and how many sick days do employees get?"
q_multi_emb = model.encode(q_multi, normalize_embeddings=True)
for k in [1, 3]:
    r = collection.query(query_embeddings=[q_multi_emb.tolist()], n_results=k,
                         include=["documents"])
    print(f"🎯 K={k}:")
    for d in r["documents"][0]:
        print("   ", d[:70])
print("\n💡 With K=1 only one fact is available; K=3 supplies both. K is a recall/precision trade-off.")



### 🧪 Experiment 5 — Top-K too large
Cramming every chunk into the prompt adds noise (and tokens).


In [ ]:

for k in [1, len(chunks)]:
    r = collection.query(query_embeddings=[q_emb.tolist()], n_results=k, include=["documents"])
    p = build_prompt(QUERY, [(d, {"source": "leave_policy.pdf", "page": 1}) for d in r["documents"][0]],
                     max_chunks=k)
    print(f"🎯 K={k}: prompt = {len(p)} chars, ~{len(tokenizer.tokenize(p))} tokens — context chunks: {k}")
print("\n💡 Larger K = more tokens, more cost, and irrelevant chunks that can distract the LLM")
print("   (e.g., the Work-From-Home chunk is noise for a maternity question).")



### 🧪 Experiment 6 — Re-ranking changes the ordering
A query whose best answer requires understanding "new parent", not just word overlap.


In [ ]:

q_parent = "How much time off does a new parent get?"
q_parent_emb = model.encode(q_parent, normalize_embeddings=True)
r = collection.query(query_embeddings=[q_parent_emb.tolist()], n_results=5,
                     include=["documents", "distances"])
docs, dists = r["documents"][0], r["distances"][0]
ce = cross.predict([[q_parent, d] for d in docs])

print("🎯 BEFORE re-ranking (cosine):")
for rank, i in enumerate(np.argsort(dists), start=1):
    print(f"   {rank}. sim={1 - dists[i]:.3f} | {docs[i][:55]}")
print("\n🎯 AFTER re-ranking (cross-encoder):")
for rank, i in enumerate(np.argsort(-ce), start=1):
    print(f"   {rank}. score={ce[i]:.3f} | {docs[i][:55]}")
print("\n💡 The cross-encoder reads 'new parent' TOGETHER with each chunk, so it can promote the")
print("   maternity/paternity chunks that cosine ranking under-ranked.")



## 🎤 Interview checkpoint
**Q: What are common failure modes of RAG, and how do you fix them?**
**A:** Poor chunking (tune size/overlap), retrieval misses (better embeddings/hybrid search),
wrong K (tune recall vs noise), irrelevant context (re-ranking, filters, thresholds), and
hallucination (strict prompting + "don't know" fallback + citations).



---

# 📌 SECTION 22 — RAG VS FINE-TUNING

```
RAG                                    FINE-TUNING
documents (external, editable)         training data (baked into weights)
     ↓                                     ↓
retrieval (search at query time)       optimization (gradient descent)
     ↓                                     ↓
context injected into prompt           updated model parameters
     ↓
LLM answers from fresh context
```

| | RAG | Fine-tuning |
|---|---|---|
| Where knowledge lives | External docs / vector DB | Inside model weights |
| Update knowledge | Edit the document — instant | Re-train — hours/days + cost |
| Best for | Factual, changing, verifiable knowledge | Style, tone, format, specialized behaviour |
| Explainability | Citations possible | Hard (weights are opaque) |
| Risk | Retrieval misses or noise | Hallucination from memorized (stale) facts |

They are **complements**, not rivals: production systems often fine-tune for behaviour AND use
RAG for knowledge.



---

# 📌 SECTION 23 — PROJECT ARCHITECTURE (the two phases)

## 🕐 INGESTION TIME (offline, once per document)
```
PDF
 ↓  Section 2   PyMuPDF extraction
text (+ metadata: source, page)
 ↓  Section 3   chunking with overlap
chunks
 ↓  Section 7   real embedding model (MiniLM, 384-d)
chunk vectors
 ↓  Section 14  store in ChromaDB
vector database  (vectors + text + metadata)
```

## 🕑 QUERY TIME (online, per question)
```
User Query
 ↓  Section 12  embed with the SAME model
query vector
 ↓  Section 15  vector search (cosine)
top-K chunks
 ↓  Section 16  cross-encoder re-ranking
best chunks
 ↓  Section 17  build prompt (system + context + question)
prompt
 ↓  Section 18  LLM
answer (+ source citation)
```

## 💻 The entire pipeline as ONE function (the real thing, not a diagram)


In [ ]:

def run_rag(query, k=5, rerank_top=3, max_chunks=3, llm_key="",
            llm_base="https://api.openai.com/v1", llm_model="gpt-4o-mini"):
    """Complete RAG pipeline: query -> answer + sources. One function, zero magic."""
    # 1. embed the query with the SAME model
    qv = model.encode(query, normalize_embeddings=True)

    # 2. vector search (cosine space)
    res = collection.query(query_embeddings=[qv.tolist()], n_results=min(k, collection.count()),
                           include=["documents", "metadatas", "distances"])
    docs, metas, dists = res["documents"][0], res["metadatas"][0], res["distances"][0]

    # 3. re-rank the candidates with the cross-encoder
    if len(docs) > 1 and cross is not None:
        ce = cross.predict([[query, d] for d in docs])
        order = np.argsort(-ce)
    else:
        order = list(range(len(docs)))
    ranked = [(docs[i], metas[i]) for i in order]

    # 4. build the prompt from the best chunks
    prompt = build_prompt(query, ranked, max_chunks=max_chunks)

    # 5. generate (real LLM if a key is provided, otherwise clearly-labelled demo)
    if llm_key:
        answer = call_openai_compatible(prompt, llm_key, base_url=llm_base, model_name=llm_model)
    else:
        answer = (f"DEMO MODE (not a real LLM answer) — best retrieved chunk: "
                  f"\"{ranked[0][0]}\"")
    return {"answer": answer, "prompt": prompt, "sources": ranked[:max_chunks],
            "retrieval_sims": [round(1 - d, 4) for d in dists]}

# ✅ END-TO-END SMOKE TEST — proves the whole notebook works from ingestion to answer.
_llm_key = (os.environ.get("LLM_API_KEY", "").strip() or api_key or "")
smoke = run_rag(QUERY, k=5, rerank_top=3, max_chunks=3,
                llm_key=_llm_key, llm_base=base_url, llm_model=llm_model)
print("=" * 70)
print("QUESTION:", QUERY)
print("\nANSWER:", smoke["answer"])
print("\nTOP SOURCE:", f"{smoke['sources'][0][1]['source']} — page {smoke['sources'][0][1]['page']}")
print("  (", smoke["sources"][0][0], ")")
print("\nRETRIEVAL SIMILARITIES:", smoke["retrieval_sims"])
print("=" * 70)
print("✅ FULL PIPELINE WORKS: PDF -> text -> chunks -> tokens -> embeddings -> vector DB")
print("   -> query embedding -> search -> re-rank -> prompt -> answer.")



---

# 📌 SECTION 24 — CODE QUALITY & WHAT LANGCHAIN WOULD HIDE

## Clean-code choices made in this notebook
- One idea per cell; sections run top-to-bottom with **no hidden state**.
- `chunk_text`, `extract_pdf`, `build_prompt`, `run_rag` — small, named, reusable functions.
- Shapes printed at every transformation (`(5, 384)`, `(1, T, 384)`…) so you can always check your mental model.
- Toy math **labelled**, real model output **labelled** — never mixed.
- Idempotent ingestion (re-running a cell doesn't duplicate data).

## How LangChain abstracts exactly what we built
| What we wrote by hand | LangChain abstraction |
|---|---|
| `extract_pdf()` with PyMuPDF | `PyMuPDFLoader` |
| `chunk_text(size, overlap)` | `RecursiveCharacterTextSplitter(chunk_size, chunk_overlap)` |
| `model.encode(text)` | `HuggingFaceEmbeddings` |
| `collection.add(...)` / `collection.query(...)` | `Chroma` vector store |
| `build_prompt(...)` | `PromptTemplate` |
| retrieval + re-rank + LLM | `RetrievalQA` / LCEL chains |

Under the hood they do exactly the same operations you just implemented. Now you know what's inside.


In [ ]:

# Side-by-side: the same pipeline in ~15 lines of LangChain vs our ~200 lines of manual code.
# This cell is OPTIONAL — it only runs if langchain is installed.

try:
    from langchain_community.document_loaders import PyMuPDFLoader
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain.chains import RetrievalQA
    from langchain.prompts import PromptTemplate

    print("LangChain is installed — running the side-by-side comparison.\n")

    # 1. Load PDF (we wrote: fitz.open -> page.get_text)
    loader = PyMuPDFLoader(pdf_path)
    lc_docs = loader.load()

    # 2. Chunk (we wrote: chunk_text with sliding window)
    splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
    lc_chunks = splitter.split_documents(lc_docs)

    # 3. Embed + store (we wrote: model.encode -> collection.add)
    embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)
    lc_vectordb = Chroma.from_documents(lc_chunks, embeddings)

    # 4. Retrieve + generate (we wrote: collection.query -> build_prompt -> call_openai_compatible)
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template=("""Answer using ONLY the supplied context.
If the answer is not present, say you don't know.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:""")
    )
    qa_chain = RetrievalQA.from_chain_type(
        llm=None,  # no LLM here — just show the retrieved docs
        retriever=lc_vectordb.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt_template},
    )

    # Retrieve (without LLM generation) to compare retrieval results
    # We use the retriever directly
    lc_results = lc_vectordb.similarity_search_with_score(QUERY, k=3)
    print("LangChain retrieval results (same query, same embedding model):")
    for doc, score in lc_results:
        print(f"  score={score:.4f} | {doc.page_content[:60]}... | {doc.metadata}")

    print()
    print("OUR manual retrieval results:")
    for rank, (doc, meta, dist) in enumerate(zip(results["documents"][0],
                                                 results["metadatas"][0],
                                                 results["distances"][0]), start=1):
        sim = 1.0 - dist
        print(f"  sim={sim:.4f} | {doc[:60]}... | {meta['source']} p.{meta['page']}")

    print()
    print("Same numbers, same result. LangChain saves ~150 lines of boilerplate,")
    print("but now you know exactly what those 150 lines do.")

except ImportError:
    print("LangChain is not installed — skipping the side-by-side demo.")
    print("To try it: pip install langchain langchain-community langchain-huggingface")



---

# 📌 SECTION 25 — INTERVIEW CHECKPOINT + FINAL CHEAT SHEET + GLOSSARY

## 🎤 Rapid-fire interview questions (all answered in this notebook)
**Q: What is an embedding?**
**A:** A learned numerical representation of text in a vector space where semantic relationships
can be compared mathematically (e.g., via cosine similarity).

**Q: What is a token ID?**
**A:** An integer index assigned to a token by the tokenizer — a vocabulary position, not a semantic value.

**Q: Where do embedding values come from?**
**A:** They are learned model parameters, optimized by gradient descent during training
(`E_new = E_old − lr · dL/dE`).

**Q: What is cosine similarity?**
**A:** `(q·c)/(||q||·||c||)` — directional similarity between two vectors, in [−1, 1].

**Q: Why use a vector database?**
**A:** To store vectors with their payloads and search them efficiently (ANN) at scale.

**Q: Why re-rank?**
**A:** First-stage vector retrieval is fast but coarse; a cross-encoder sees query+chunk together
and re-orders the small candidate set with much higher precision.

**Q: What is chunk overlap for?**
**A:** To keep phrases that straddle chunk boundaries intact in at least one chunk.

**Q: RAG vs fine-tuning?**
**A:** RAG injects retrieved, editable knowledge at query time (grounding, citations); fine-tuning
bakes knowledge into weights (style/behaviour, costly to update).

## 🏁 FINAL CHEAT SHEET
```
INGESTION                               QUERY
PDF                                     question
 ↓ text extraction (PyMuPDF)             ↓ query embedding (same model)
chunks (+overlap)                        ↓ cosine similarity vs stored vectors
 ↓ tokens → token IDs (vocab lookup)     ↓ top-K retrieval from vector DB
 ↓ embedding matrix row lookup           ↓ cross-encoder re-ranking
 ↓ Transformer + self-attention          ↓ best context
 ↓ contextual token vectors              ↓ prompt (system + context + question)
 ↓ mean pooling + normalize              ↓ LLM
 ↓ chunk embedding (384-d)               ↓ answer + source citation
 ↓ store in vector DB
```



## 📚 GLOSSARY

| Term | Simple definition |
|---|---|
| Document | A source file (e.g., PDF) containing the knowledge we want to retrieve. |
| Text extraction | Pulling plain text out of a binary format (PDF) with a parser like PyMuPDF. |
| Chunk | A short, self-contained piece of text used as a retrieval unit. |
| Chunk size | Number of tokens/words per chunk. Trade-off between context and focus. |
| Chunk overlap | Words repeated between adjacent chunks so boundary phrases stay intact. |
| Token | The atomic text piece the model operates on (a word or a subword like `##ably`). |
| Tokenizer | Deterministic function text → tokens (rules + vocabulary). |
| Token ID | Integer index of a token in the vocabulary. A position, not a meaning. |
| Vocabulary | The fixed ordered list of tokens the model knows (MiniLM: 30,522). |
| Embedding | A learned dense vector representing text, where similarity ≈ semantic similarity. |
| Embedding dimension | Size of the embedding vector (MiniLM: 384). |
| Embedding matrix | Learned table `(vocab_size × dim)`; each row is one token's vector. |
| Token embedding | The static dictionary vector of a token (before seeing context). |
| Hidden state | Intermediate vector produced inside the network layers. |
| Contextual representation | A token vector after attention has mixed in the other tokens of the sentence. |
| Transformer | Stack of self-attention + feed-forward layers that builds contextual representations. |
| Self-attention | Each token gathers information from all tokens, weighted by compatibility. |
| Q, K, V | Query, Key, Value projections in attention: `softmax(QKᵀ/√d_k)·V`. |
| Pooling | Aggregating token vectors into one vector (MiniLM uses mean pooling). |
| Chunk embedding | The single fixed-size vector representing a whole chunk (after pooling + normalize). |
| Vector | A point/direction in space; a list of numbers with defined operations. |
| Vector space | The mathematical space where distances/angles encode similarity. |
| Cosine similarity | `(q·c)/(||q||·||c||)` — directional similarity between two vectors. |
| Dot product | Sum of element-wise products: `Σ qᵢcᵢ` (numerator of cosine). |
| Vector database | Storage + fast nearest-neighbour search over vectors with payloads (ChromaDB). |
| Retriever | The component that returns candidate chunks for a query (bi-encoder + ANN). |
| Top-K | The K highest-ranked candidates returned by retrieval. |
| Re-ranker | A second-stage model (cross-encoder) that re-orders the candidates precisely. |
| Cross-encoder | Model that scores `query+chunk` jointly via full cross-attention. |
| Context | The retrieved chunks inserted into the prompt for grounding. |
| Prompt | Instructions + context + question sent to the LLM. |
| LLM | The generative model producing the final answer. |
| RAG | Retrieval-Augmented Generation: retrieve → augment prompt → generate grounded answer. |
| Hallucination | The LLM producing plausible but unsupported text. |
| Grounding | Forcing the answer to be based on provided evidence (with citations). |

## ✅ You can now explain the entire journey in an interview:
```
"Female employees receive 26 weeks of paid maternity leave."
 → tokenizer → 10 tokens → 10 token IDs → embedding-matrix row lookups
 → 12 Transformer layers of self-attention → contextual token vectors
 → mean pooling + normalization → one 384-d chunk embedding → vector DB.

"How many weeks of maternity leave are available for a female worker?"
 → same model → query vector → cosine similarity vs chunks → top-K
 → cross-encoder re-ranking → best chunks → prompt → LLM → answer + citation.
```
